# 08 — LLM-Assisted Reviewer Copilot

Demonstrate grounded LLM reviewer notes, data Q&A, scenario summaries, and rule explanations.

All outputs are labeled **AI-generated recommendation — human review required**.
Prompts, model, timestamp, and hallucination flags are logged to `logs/llm_interaction_log.jsonl`.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from src.llm_copilot.reviewer import ReviewerCopilot
from src.modeling.anomaly import AnomalyDetector
from src.features.engineer import LeakageSafeFeatureEngineer
from src.utils.config import get_settings

config = get_settings('../config/settings.yaml')
train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
servicer_df = pd.read_csv('../data/synthetic/servicer_updates.csv')
for col in ['reporting_month','origination_month']:
    if col in train_df.columns:
        train_df[col] = pd.to_datetime(train_df[col]).dt.to_period('M')
from src.pipeline.loader import reconcile_servicer_updates
train_df, _ = reconcile_servicer_updates(train_df, servicer_df)

fe = LeakageSafeFeatureEngineer(config)
fe.load_artifacts('../models/feature_engineering')
detector = AnomalyDetector(config)
detector.load_artifacts('../models/anomaly')
anomaly_result = detector.detect(train_df)

copilot = ReviewerCopilot(config)
print('Copilot ready. LLM API key set:', bool(copilot.client.api_key))


## 1. Reviewer Note for a Flagged Loan

In [ ]:
flagged_idx = np.where(anomaly_result.flags == 1)[0][0]
flagged_row = train_df.iloc[flagged_idx]
X_f = fe.transform(train_df.iloc[[flagged_idx]])
model_def = joblib.load('../models/classification/next_12m_default_flag_improved.pkl')
model_dq  = joblib.load('../models/classification/next_3m_delinquency_flag_improved.pkl')
model_outputs = {
    'next_12m_default_flag':        float(model_def.predict_proba(X_f)[0, 1]),
    'next_3m_delinquency_flag':     float(model_dq.predict_proba(X_f)[0, 1]),
}
shap_drivers = anomaly_result.drivers[flagged_idx] if anomaly_result.drivers[flagged_idx] else [
    {'feature': 'days_past_due', 'contribution': 0.45},
    {'feature': 'balance_ratio', 'contribution': 0.28},
]

note = copilot.generate_reviewer_note(
    str(flagged_row['loan_id']), model_outputs, shap_drivers, ['R014'], 'data_quality'
)
print('REVIEWER NOTE:')
print(note)


## 2. Data Dictionary Q&A

In [ ]:
question = "What does days_past_due mean and when should it trigger a review?"
answer = copilot.answer_data_question(question)
print(f'Q: {question}')
print(f'A: {answer}')


## 3. Scenario Summary

In [ ]:
scenario_json = json.load(open('../reports/scenario/scenario_results.json'))
class SR:
    def __init__(self, a, s): self.aggregate_projections = a; self.segment_projections = s
scene_objs = {k: SR(v.get('aggregate_projections',{}), v.get('segment_projections',{})) for k,v in scenario_json.items()}
summary = copilot.summarize_scenario(scene_objs)
print('SCENARIO SUMMARY:')
print(summary)


## 4. Validation Rule Explanation

In [ ]:
rule_exp = copilot.explain_validation_rule('R014')
print('RULE EXPLANATION:')
print(rule_exp)


## 5. Rejected / Corrected Examples (Governance Evidence)

In [ ]:
rejected = json.load(open('../reports/copilot/rejected_examples.json'))
for i, ex in enumerate(rejected, 1):
    print(f'\n--- Rejected Example {i}: {ex["interaction_type"]} ---')
    print(f'Bad output (truncated):  {str(ex["bad_output"])[:200]}')
    print(f'Problems: {ex["bad_output_problems"][0]}')
    print(f'Corrected (truncated):   {str(ex["corrected_output"])[:200]}')
    print(f'Lesson: {ex["lesson"]}')


## 6. Interaction Log

In [ ]:
log_path = Path('../logs/llm_interaction_log.jsonl')
if log_path.exists():
    logs = [json.loads(l) for l in log_path.read_text().strip().split('\n') if l]
    print(f'Total logged interactions: {len(logs)}')
    for entry in logs[-4:]:
        print(f'  [{entry["timestamp"]}] type={entry.get("label","?")}  flagged={entry.get("flagged",False)}  model={entry.get("model","?")}')
else:
    print('No log file found — run copilot cells above first.')
